In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

from tissuenarrator.llm import VLLMWrapper
from tissuenarrator.evaluator import SpatialEvaluator

/home/sizheliu/miniconda3/envs/unsloth/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 03-26 07:39:35 [__init__.py:241] Automatically detected platform cuda.


### Run this once if LoRA weights are not merged yet

In [ ]:
from unsloth import FastLanguageModel

checkpoint_path = "merfish_epoch3_lora" # Path to the LoRA adapter checkpoint
max_seq_length = 32000

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = checkpoint_path,
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    load_in_8bit = False,
)
model.save_pretrained_merged("merfish_epoch3_merged", tokenizer, save_method = "merged_16bit",)

### Load merged model with vLLM for inference

In [2]:
MODEL_PATH = "merfish_epoch3_merged"
model = VLLMWrapper(MODEL_PATH, max_length=32000)

INFO 03-26 07:39:36 [utils.py:326] non-default args: {'model': 'merfish_epoch3_merged', 'max_model_len': 32000, 'disable_log_stats': True}
INFO 03-26 07:39:44 [__init__.py:711] Resolved architecture: Qwen3ForCausalLM
INFO 03-26 07:39:44 [__init__.py:1750] Using max model len 32000


2026-03-26 07:39:45,339	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 03-26 07:39:45 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 03-26 07:39:45 [__init__.py:2921] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 03-26 07:39:52 [__init__.py:241] Automatically detected platform cuda.
(EngineCore_0 pid=1389826) INFO 03-26 07:39:53 [core.py:636] Waiting for init message from front-end.
(EngineCore_0 pid=1389826) INFO 03-26 07:39:53 [core.py:74] Initializing a V1 LLM engine (v0.10.1.1) with config: model='merfish_epoch3_merged', speculative_config=None, tokenizer='merfish_epoch3_merged', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32000, download_dir=None, load_format=auto, tensor_parallel

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:00<00:00,  2.28it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.57it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.64it/s]
(EngineCore_0 pid=1389826) 


(EngineCore_0 pid=1389826) INFO 03-26 07:39:56 [default_loader.py:262] Loading weights took 1.49 seconds
(EngineCore_0 pid=1389826) INFO 03-26 07:39:57 [gpu_model_runner.py:2007] Model loading took 7.5532 GiB and 1.677392 seconds
(EngineCore_0 pid=1389826) INFO 03-26 07:40:06 [backends.py:548] Using cache directory: /home/sizheliu/.cache/vllm/torch_compile_cache/36c77c3634/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_0 pid=1389826) INFO 03-26 07:40:06 [backends.py:559] Dynamo bytecode transform time: 9.55 s
(EngineCore_0 pid=1389826) INFO 03-26 07:40:11 [backends.py:194] Cache the graph for dynamic shape for later use
(EngineCore_0 pid=1389826) INFO 03-26 07:40:45 [backends.py:215] Compiling a graph for dynamic shape takes 37.93 s
(EngineCore_0 pid=1389826) INFO 03-26 07:40:57 [monitor.py:34] torch.compile takes 47.48 s in total
(EngineCore_0 pid=1389826) INFO 03-26 07:40:58 [gpu_worker.py:276] Available KV cache memory: 33.69 GiB
(EngineCore_0 pid=1389826) INFO 03-26 07:40:5

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:03<00:00, 19.90it/s]


(EngineCore_0 pid=1389826) INFO 03-26 07:41:02 [gpu_model_runner.py:2708] Graph capturing finished in 4 secs, took 0.61 GiB
(EngineCore_0 pid=1389826) INFO 03-26 07:41:02 [core.py:214] init engine (profile, create kv cache, warmup model) took 65.71 seconds
INFO 03-26 07:41:03 [llm.py:298] Supported_tasks: ['generate']


### Load demo dataset

In [3]:
import pandas as pd
ds_path = "merfish_demo_sample100.parquet"
df = pd.read_parquet(ds_path)

In [4]:
df

,cell_name,prompt,answer,x,y,section,class,split,neighbor_cell_names,spatial_domain
0,202557446572192392748730476427011253583,"<pos> X: 4644, Y: 5952 <meta> class: Vascular,...","<pos> X: 4768, Y: 6093 <meta> class: HY GABA, ...",4768.434927,6093.302439,Zhuang-ABCA-1.084,HY GABA,test,"[107863989153803449591883120455390268715, 1082...",HY
1,227264332012171020156464499035405844802,"<pos> X: 4908, Y: 6010 <meta> class: Vascular,...","<pos> X: 5044, Y: 6053 <meta> class: MY GABA, ...",5044.106303,6053.035240,Zhuang-ABCA-1.121,MY GABA,test,"[206802976736754721851066348858123947277, 2937...",MY
2,337930275008977742398263331304590404212,"<pos> X: 2822, Y: 1724 <meta> class: Astro-Epe...","<pos> X: 2982, Y: 1736 <meta> class: CTX-CGE G...",2982.235261,1736.345754,Zhuang-ABCA-1.060,CTX-CGE GABA,test,"[172644761415763159674982298243901362705, 1767...",Isocortex
3,24056441603073228864071521440735312169,"<pos> X: 2690, Y: 4879 <meta> class: OPC-Oligo...","<pos> X: 2888, Y: 4892 <meta> class: CNU-HYa G...",2888.267236,4892.777064,Zhuang-ABCA-1.128,CNU-HYa GABA,test,"[164046837872914046198943224972204787679, 1843...",CB
4,239122422756298823808269348545484438938,"<pos> X: 5084, Y: 2603 <meta> class: OPC-Oligo...","<pos> X: 5206, Y: 2742 <meta> class: MB GABA, ...",5206.203487,2742.781585,Zhuang-ABCA-1.119,MB GABA,test,"[240769283068045641019709574189782965302, 1141...",MB
...,...,...,...,...,...,...,...,...,...,...
95,117848000252752629528340094032669016812,"<pos> X: 3205, Y: 5190 <meta> class: Astro-Epe...","<pos> X: 3389, Y: 5248 <meta> class: CB Glut, ...",3389.138961,5248.226579,Zhuang-ABCA-1.125,CB Glut,test,"[229920903406150769580094442809544054323, 1048...",cm
96,208742976724299269274749384504652507698,"<pos> X: 4977, Y: 2860 <meta> class: DG-IMN Gl...","<pos> X: 5104, Y: 3006 <meta> class: MH-LH Glu...",5104.966102,3006.891752,Zhuang-ABCA-1.084,MH-LH Glut,test,"[89317035059603740360564037571371893627, 19682...",TH
97,15135970301494062861642031671733013811,"<pos> X: 4031, Y: 2612 <meta> class: OPC-Oligo...","<pos> X: 4205, Y: 2633 <meta> class: CB GABA, ...",4205.315084,2633.363038,Zhuang-ABCA-1.122,CB GABA,test,"[229031507992220129749005501330699950220, 9790...",CB
98,110844604594402818862846923800916891099,"<pos> X: 3057, Y: 4397 <meta> class: OPC-Oligo...","<pos> X: 3206, Y: 4525 <meta> class: MY GABA, ...",3206.520741,4525.797844,Zhuang-ABCA-1.125,MY GABA,test,"[25297531291011473182046948099995230809, 53769...",CB


### Inference Modes
- **meta_all** — Uses both the neighbor sentence and the target cell’s metadata.  
- **meta_neighbor** — Uses only the neighbor sentence’s metadata.  
- **pos_only** — Excludes all metadata.


In [5]:
per_row_model, overall_model, gen_blocks = SpatialEvaluator(model).evaluate_prompt_df(
    df,
    max_rows=100,
    top_k_list=(10, 30, 50),
    greedy=False,
    batch_size=50,
    temperature=0.3,
    top_k=10,
    max_new_tokens=400,
    mode="meta_all",
    total_prompt_cells=30,
)

Adding requests: 100%|██████████| 50/50 [00:01<00:00, 44.48it/s]

Adding requests: 100%|██████████| 50/50 [00:01<00:00, 45.04it/s]

100%|██████████| 100/100 [05:21<00:00,  3.21s/it]


In [6]:
print(overall_model)

{'ndcg': np.float64(0.9188225514033218), 'top_10_overlap': np.float64(0.4700000000000001), 'top_10_ndcg': np.float64(0.8828960888030182), 'top_30_overlap': np.float64(0.48066666666666663), 'top_30_ndcg': np.float64(0.8911749761506925), 'top_50_overlap': np.float64(0.4824000000000001), 'top_50_ndcg': np.float64(0.8981422281797573)}


In [7]:
print(gen_blocks[0])

<pos> X: 4768, Y: 6093 <meta> class: HY GABA, spatial_domain: HY <cs> GDA DLK1 RREB1 BAIAP3 CALB1 ECEL1 OTP GAD2 PTK2B SV2B SLC32A1 MEF2C EPHA4 CNIH3 IRS4 ADGRL2 ADCY2 CRYM NTRK3 OPRK1 PDE1A PTHLH ZIM1 LDB2 RPRM EPHA7 TRPC4 SSTR1 TSHZ2 ISL1 AMIGO2 RGS12 ARHGAP36 EPHA10 PCP4L1 GRM8 EPHA6 GRIK3 GABBR2 SLC38A1 LYPD6B KCNMB2 SLITRK6 BCL11A FILIP1 KCTD8 IGHM SYT6 ZFP536 GPC4 FSTL5 GPRC5B VWC2 EGFEM1 HRH3 NGB KCNG1 TOX2 SEMA5A BCL11B SNTB1 GRM3 ZFP831 GRIK4 CACNG5 PTH2R TRPC7 RASGRP2 TACR1 CHRM1 ANKRD63 PCDH8 SIX3 EPHA3 RSKR GUCY1A1 OPRM1 CRHR1 HTR1B GLRA3 KCNJ4 LAMP5 DDC ERBB4 ZBTB16 CNR1 CHRNA7 NELL1 GSTO1 FGF11 SSTR2 HTR1A RGS6 ZMAT4 SEMA6A FAM107A FGF13 OTOF GPR26 TRPC5 SYT17 PCDH20 MPPED2 APEX1 LYPD6 PRDM16 STAC2 </cs>
